# Inventory Health Dashboard — SKU × Country

**Dataset:** FMCG multi-country sales (Kaggle, ~1M rows, 3 years)

**KPIs computed per (SKU × Country):**
1. **Demand stats** — total units, avg daily demand, std daily demand, CV
2. **Revenue stats** — total revenue, % of country revenue
3. **ABC segmentation** — by revenue (Pareto: A=top 70%, B=next 20%, C=last 10%)
4. **XYZ segmentation** — by demand variability (X<0.5, Y=0.5-1.0, Z>1.0)
5. **Safety Stock** — SS = Z × σ_daily × √lead_time   (Z=1.65 → 95% service level)
6. **Reorder Point** — ROP = (μ_daily × lead_time) + SS
7. **Current health** — stock_on_hand, days_of_supply, stockout_rate, status flag

## 1. Load + basic inspection

In [23]:
# !pip install jupytext --To sync Jupyter notebooks with Python script files
# !jupytext --set-formats ipynb,py inventory_health_dashboard.ipynb -- To convert Jupyter notebook to Python script and vice versa

In [24]:
# ============================================================
# CONFIGURATION — change these values as needed
# ============================================================

# ABC segmentation thresholds (cumulative revenue %)
ABC_THRESHOLDS = {
    "A": 0.70,   # top 70% of revenue
    "B": 0.90,   # next 20% (cumulative 90%)
    # C = remaining 10%
}

# XYZ segmentation thresholds (coefficient of variation)
XYZ_THRESHOLDS = {
    "X": 0.50,   # CV ≤ 0.50 → steady demand
    "Y": 1.00,   # CV ≤ 1.00 → moderate variability
    # Z = above 1.0 → erratic
}

# Service levels by ABC (Z-score for normal distribution)
ABC_SERVICE_LEVELS = {
    "A": 2.33,   # 99% service level
    "B": 1.65,   # 95% service level
    "C": 1.28,   # 90% service level
}

# Health status thresholds
STATUS_THRESHOLDS = {
    "low_multiplier":      1.5,   # current_stock ≤ 1.5 × ROP → LOW
    "overstock_multiplier": 4.0,  # current_stock > 4.0 × ROP → OVERSTOCK
}

In [25]:
import numpy as np
import pandas as pd


df = pd.read_csv("E:/Inventory Optimization/data/sales_data.csv")
df.head()


,date,year,month,day,weekofyear,weekday,is_weekend,is_holiday,temperature,rain_mm,...,discount_pct,promo_flag,gross_sales,net_sales,stock_on_hand,stock_out_flag,lead_time_days,supplier_id,purchase_cost,margin_pct
0,01-01-2021,2021,1,1,53,4,0,1,8.44,1.24,...,0.1,1,167.84,151.06,248,0,11,S008,7.53,0.182
1,02-01-2021,2021,1,2,53,5,1,0,12.61,1.12,...,0.0,0,125.88,125.88,238,0,6,S057,5.19,0.505
2,03-01-2021,2021,1,3,53,6,1,0,12.02,2.69,...,0.3,1,398.62,279.03,238,0,6,S017,5.59,0.168
3,04-01-2021,2021,1,4,1,0,0,0,7.76,4.65,...,0.0,0,83.92,83.92,216,0,7,S012,7.81,0.255
4,05-01-2021,2021,1,5,1,1,0,0,11.16,1.77,...,0.2,1,178.33,142.66,372,0,8,S038,7.62,0.073


In [26]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1048575 entries, 0 to 1048574
Data columns (total 33 columns):
 #   Column          Non-Null Count    Dtype  
---  ------          --------------    -----  
 0   date            1048575 non-null  str    
 1   year            1048575 non-null  int64  
 2   month           1048575 non-null  int64  
 3   day             1048575 non-null  int64  
 4   weekofyear      1048575 non-null  int64  
 5   weekday         1048575 non-null  int64  
 6   is_weekend      1048575 non-null  int64  
 7   is_holiday      1048575 non-null  int64  
 8   temperature     1048575 non-null  float64
 9   rain_mm         1048575 non-null  float64
 10  store_id        1048575 non-null  str    
 11  country         1048575 non-null  str    
 12  city            1048575 non-null  str    
 13  channel         1048575 non-null  str    
 14  latitude        1048575 non-null  float64
 15  longitude       1048575 non-null  float64
 16  sku_id          1048575 non-null  str    
 17  

In [27]:
df.date = pd.to_datetime(df.date, format="%d-%m-%Y")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1048575 entries, 0 to 1048574
Data columns (total 33 columns):
 #   Column          Non-Null Count    Dtype         
---  ------          --------------    -----         
 0   date            1048575 non-null  datetime64[us]
 1   year            1048575 non-null  int64         
 2   month           1048575 non-null  int64         
 3   day             1048575 non-null  int64         
 4   weekofyear      1048575 non-null  int64         
 5   weekday         1048575 non-null  int64         
 6   is_weekend      1048575 non-null  int64         
 7   is_holiday      1048575 non-null  int64         
 8   temperature     1048575 non-null  float64       
 9   rain_mm         1048575 non-null  float64       
 10  store_id        1048575 non-null  str           
 11  country         1048575 non-null  str           
 12  city            1048575 non-null  str           
 13  channel         1048575 non-null  str           
 14  latitude        1048575 non-n

In [28]:
# start with Italy data for testing

italy = df[df["country"] == "Italy"]

# SKU sold in multiple stores on same day
multi_store = (
    italy.groupby(["date", "sku_id"])["store_id"]
           .nunique()
           .reset_index(name="n_stores")
)

multi_store = multi_store[multi_store["n_stores"] > 1]

# Get required rows and columns
result = italy.merge(
    multi_store[["date", "sku_id"]],
    on=["date", "sku_id"],
    how="inner"
)[["date", "store_id", "sku_id", "units_sold"]]

result.sort_values(["date", "sku_id", "store_id"]).head()


#So the data in the dataset is at the store level, but we need to analyze it at the country level.
# We will aggregate the data by date, country, and SKU to get the total units sold per day for each SKU in France.

,date,store_id,sku_id,units_sold
62415,2021-01-01,STORE0002,SKU0001,56
101835,2021-01-01,STORE0003,SKU0001,80
196005,2021-01-01,STORE0010,SKU0001,103
264990,2021-01-01,STORE0011,SKU0001,78
58035,2021-01-01,STORE0002,SKU0002,61


In [29]:
#Pick only the relevant columns to speed up loading and reduce memory usage

USECOLS = [
    "date", "month", "year", "country", "sku_id", "sku_name", "category", "brand",
    "units_sold", "net_sales", "stock_on_hand", "stock_out_flag",
    "lead_time_days", "purchase_cost",
]

df = df[USECOLS].copy()
df.head()



,date,month,year,country,sku_id,sku_name,category,brand,units_sold,net_sales,stock_on_hand,stock_out_flag,lead_time_days,purchase_cost
0,2021-01-01,1,2021,Germany,SKU0086,BrandB Shampoo,Personal Care,BrandB,16,151.06,248,0,11,7.53
1,2021-01-02,1,2021,Germany,SKU0086,BrandB Shampoo,Personal Care,BrandB,12,125.88,238,0,6,5.19
2,2021-01-03,1,2021,Germany,SKU0086,BrandB Shampoo,Personal Care,BrandB,38,279.03,238,0,6,5.59
3,2021-01-04,1,2021,Germany,SKU0086,BrandB Shampoo,Personal Care,BrandB,8,83.92,216,0,7,7.81
4,2021-01-05,1,2021,Germany,SKU0086,BrandB Shampoo,Personal Care,BrandB,17,142.66,372,0,8,7.62


## 2. Aggregate to SKU × Country × Day
Multiple stores per country exist. We sum to one row per (sku, country, day).

In [30]:
#Daily aggregation at sku and country level

df.drop_duplicates(subset=["date", "country", "sku_id"], keep="first", inplace=True)  # remove duplicates if any

daily_kpi = (
    df.groupby(["sku_id", "country", "date"], observed=True)
      .agg(
          units_sold     = ("units_sold", "sum"),
          net_sales      = ("net_sales", "sum"),
          stock_on_hand  = ("stock_on_hand", "sum"),
          stockouts      = ("stock_out_flag", "max"),  # if any store had stockout, it's a stockout day for that SKU-country
          n_store_days   = ("stock_out_flag", "size"),
          lead_time_days = ("lead_time_days", "mean"),
          purchase_cost  = ("purchase_cost", "mean"),
      )
      .reset_index()
)
print(f"Daily SKU × Country rows: {len(daily_kpi):,}")
daily_kpi.head()

daily_kpi.to_excel("E:/Inventory Optimization/output/daily_kpi_data.xlsx", index=False)
daily_kpi.head()


Daily SKU × Country rows: 591,300


,sku_id,country,date,units_sold,net_sales,stock_on_hand,stockouts,n_store_days,lead_time_days,purchase_cost
0,SKU0001,Austria,2021-01-01,104,648.96,327,0,1,6.0,2.92
1,SKU0001,Austria,2021-01-02,258,1287.94,274,0,1,8.0,2.98
2,SKU0001,Austria,2021-01-03,367,1946.57,363,0,1,9.0,4.26
3,SKU0001,Austria,2021-01-04,230,1219.92,214,0,1,6.0,4.44
4,SKU0001,Austria,2021-01-05,105,655.20,346,0,1,8.0,4.01


## 3. KPI table per SKU × Country - Month-Year

In [31]:
# Pull the latest snapshot per SKU x Country (most recent stock_on_hand)

# Convert date to datetime (format: DD-MM-YYYY)
daily_kpi["date"] = pd.to_datetime(daily_kpi["date"], format="%d-%m-%Y")
daily_kpi["year_month"] = daily_kpi["date"].dt.to_period("M")

latest = (
    daily_kpi.sort_values("date")
         .groupby(["sku_id", "country",'year_month'], observed=True)
         .tail(1)[["sku_id", "country", "year_month", "stock_on_hand"]]
         .rename(columns={"stock_on_hand": "current_stock"})
)

monthly_kpi = (
    daily_kpi.groupby(["sku_id", "country",'year_month'], observed=True)
         .agg(
             total_units    = ("units_sold", "sum"),
             total_revenue  = ("net_sales", "sum"),
             avg_daily_demand = ("units_sold", "mean"),
             std_daily_demand = ("units_sold", "std"),
             active_days    = ("units_sold", "size"),
             stockout_days  = ("stockouts", "sum"),
             lead_time_days = ("lead_time_days", "mean"),
             unit_cost      = ("purchase_cost", "mean"),
         )
         .reset_index()
         .merge(latest, on=["sku_id", "country", "year_month"], how="left")
)

# Add SKU metadata back (category, brand, name) — first non-null per SKU
meta = df[["sku_id", "sku_name", "category", "brand"]].drop_duplicates("sku_id")
monthly_kpi = monthly_kpi.merge(meta, on="sku_id", how="left")

monthly_kpi["std_daily_demand"] = monthly_kpi["std_daily_demand"].fillna(0)
monthly_kpi["cv_demand"] = (monthly_kpi["std_daily_demand"] / monthly_kpi["avg_daily_demand"]).replace([np.inf, -np.inf], np.nan).fillna(0).round(3)
monthly_kpi["stockout_rate"] = (monthly_kpi["stockout_days"] / monthly_kpi["active_days"]).round(3)

# Check for duplicates
duplicates = monthly_kpi.duplicated(subset=["sku_id", "country", "year_month"], keep=False).sum()
print(f"KPI rows: {len(monthly_kpi):,}  (= unique SKU × Country × Month)")
print(f"Duplicate rows (same sku_id, country, year_month): {duplicates}")


monthly_kpi.to_excel("E:/Inventory Optimization/output/monthly_kpi.xlsx", index=False)
monthly_kpi.head()

KPI rows: 19,440  (= unique SKU × Country × Month)
Duplicate rows (same sku_id, country, year_month): 0


,sku_id,country,year_month,total_units,total_revenue,avg_daily_demand,std_daily_demand,active_days,stockout_days,lead_time_days,unit_cost,current_stock,sku_name,category,brand,cv_demand,stockout_rate
0,SKU0001,Austria,2021-01,4695,26976.15,151.451613,70.091768,31,0,6.612903,3.852581,371,BrandA Soda,Beverages,BrandA,0.463,0.000
1,SKU0001,Austria,2021-02,3371,19920.58,120.392857,61.287627,28,1,6.857143,3.727857,257,BrandA Soda,Beverages,BrandA,0.509,0.036
2,SKU0001,Austria,2021-03,4490,26007.69,144.838710,60.173137,31,0,6.548387,3.706452,380,BrandA Soda,Beverages,BrandA,0.415,0.000
3,SKU0001,Austria,2021-04,3365,19948.97,112.166667,44.698286,30,0,6.333333,3.747000,178,BrandA Soda,Beverages,BrandA,0.398,0.000
4,SKU0001,Austria,2021-05,4583,25415.21,147.838710,84.084916,31,2,5.806452,3.814194,319,BrandA Soda,Beverages,BrandA,0.569,0.065


## KPI Calculation - SKU x COUNTRY - YEARLY

In [32]:
# Create year column from existing year_month
monthly_kpi["year"] = monthly_kpi["year_month"].dt.year


# Sort by SKU, Country, Year-Month to ensure correct order for yearly aggregation
monthly_kpi = monthly_kpi.sort_values(["sku_id", "country", "year_month"])

# Yearly KPI table
yearly_kpi = (
    monthly_kpi.groupby(["sku_id", "country", "year"], observed=True)
               .agg(
                   total_units       = ("total_units", "sum"),
                   total_revenue     = ("total_revenue", "sum"),
                   avg_daily_demand  = ("avg_daily_demand", "mean"),
                   std_daily_demand  = ("std_daily_demand", "mean"),
                   active_days       = ("active_days", "sum"),
                   stockout_days     = ("stockout_days", "sum"),
                   current_stock     = ("current_stock", "last"),
                   lead_time_days    = ("lead_time_days", "mean"),
                   unit_cost         = ("unit_cost", "mean"),
               )
               .reset_index()
)

# Recalculate yearly KPIs
yearly_kpi["cv_demand"] = (
    yearly_kpi["std_daily_demand"] /
    yearly_kpi["avg_daily_demand"]
).replace([np.inf, -np.inf], np.nan).fillna(0).round(3)

yearly_kpi["stockout_rate"] = (
    yearly_kpi["stockout_days"] /
    yearly_kpi["active_days"]
).round(3)

# Add SKU metadata
meta = df[["sku_id", "sku_name", "category", "brand"]].drop_duplicates("sku_id")

yearly_kpi = yearly_kpi.merge(meta, on="sku_id", how="left")

# Check duplicates
duplicates = yearly_kpi.duplicated(
    subset=["sku_id", "country", "year"],
    keep=False
).sum()

print(f"Yearly KPI rows: {len(yearly_kpi):,}")
print(f"Duplicate rows: {duplicates}")


yearly_kpi.to_excel("E:/Inventory Optimization/output/yearly_kpi.xlsx", index=False)
yearly_kpi.head()

Yearly KPI rows: 1,620
Duplicate rows: 0


,sku_id,country,year,total_units,total_revenue,avg_daily_demand,std_daily_demand,active_days,stockout_days,current_stock,lead_time_days,unit_cost,cv_demand,stockout_rate,sku_name,category,brand
0,SKU0001,Austria,2021,54542,321043.34,149.096358,68.957033,365,8,426,6.529852,3.708952,0.462,0.022,BrandA Soda,Beverages,BrandA
1,SKU0001,Austria,2022,53538,314863.24,146.436534,68.209359,365,12,270,6.420328,3.751144,0.466,0.033,BrandA Soda,Beverages,BrandA
2,SKU0001,Austria,2023,53287,310313.33,145.878719,71.094469,365,12,297,6.494022,3.761100,0.487,0.033,BrandA Soda,Beverages,BrandA
3,SKU0001,France,2021,54494,319414.10,149.165316,66.024241,365,10,191,6.659556,3.745666,0.443,0.027,BrandA Soda,Beverages,BrandA
4,SKU0001,France,2022,52589,310537.35,143.817556,63.012036,365,12,342,6.599910,3.735098,0.438,0.033,BrandA Soda,Beverages,BrandA


## 4. ABC segmentation (per country)
Each country gets its own Pareto: A = top 70 % of revenue, B = next 20 %, C = bottom 10 %.

In [33]:
# Sort by country and revenue first
yearly_kpi = yearly_kpi.sort_values(
    ["country", "total_revenue"], ascending=[True, False]
).reset_index(drop=True)

# Cumulative revenue % within each country — fully vectorized
yearly_kpi["_cum_rev"]   = yearly_kpi.groupby("country")["total_revenue"].cumsum()
yearly_kpi["_total_rev"] = yearly_kpi.groupby("country")["total_revenue"].transform("sum")
yearly_kpi["_cum_pct"]   = yearly_kpi["_cum_rev"] / yearly_kpi["_total_rev"]

# ABC classification — using config thresholds
yearly_kpi["abc"] = np.where(
    yearly_kpi["_cum_pct"] <= ABC_THRESHOLDS["A"], "A",
    np.where(yearly_kpi["_cum_pct"] <= ABC_THRESHOLDS["B"], "B", "C")
)

# Drop helper columns
yearly_kpi.drop(columns=["_cum_rev", "_total_rev", "_cum_pct"], inplace=True)

# Check distribution
print(yearly_kpi["abc"].value_counts().sort_index())
print("country" in yearly_kpi.columns)  # should be True
yearly_kpi.head()

abc
A    584
B    471
C    565
Name: count, dtype: int64
True


,sku_id,country,year,total_units,total_revenue,avg_daily_demand,std_daily_demand,active_days,stockout_days,current_stock,lead_time_days,unit_cost,cv_demand,stockout_rate,sku_name,category,brand,abc
0,SKU0027,Austria,2022,47884,634485.26,131.181720,60.012622,365,12,353,6.459741,8.563528,0.457,0.033,BrandC Chips,Snacks,BrandC,A
1,SKU0027,Austria,2023,47329,628585.42,129.603853,58.973336,365,9,314,6.385714,8.538277,0.455,0.025,BrandC Chips,Snacks,BrandC,A
2,SKU0027,Austria,2021,46003,618833.98,126.044489,54.619307,365,11,322,6.651203,8.403585,0.433,0.030,BrandC Chips,Snacks,BrandC,A
3,SKU0018,Austria,2021,39096,567673.92,107.068843,36.431126,365,12,310,6.423144,8.632652,0.340,0.033,BrandF Water,Beverages,BrandF,A
4,SKU0018,Austria,2022,38866,564334.32,106.388511,36.817903,365,9,162,6.483500,8.685942,0.346,0.025,BrandF Water,Beverages,BrandF,A


## 5. XYZ segmentation (demand variability)
Based on CV of daily demand. Lower CV = more predictable.

- **X**: CV ≤ 0.5  (steady, easy to forecast)
- **Y**: 0.5 < CV ≤ 1.0 (moderate variability)
- **Z**: CV > 1.0 (lumpy / erratic)

In [34]:
# XYZ classification — using config thresholds
yearly_kpi["xyz"] = "Z"

yearly_kpi.loc[
    yearly_kpi["cv_demand"] <= XYZ_THRESHOLDS["Y"], "xyz"
] = "Y"

yearly_kpi.loc[
    yearly_kpi["cv_demand"] <= XYZ_THRESHOLDS["X"], "xyz"
] = "X"

# Combine ABC and XYZ
yearly_kpi["abc_xyz"] = (
    yearly_kpi["abc"] + yearly_kpi["xyz"]
)

# Count categories — show all 9 combinations including zeros
from itertools import product

all_combinations = ["".join(x) for x in product(["A","B","C"], ["X","Y","Z"])]

yearly_kpi["abc_xyz"].value_counts().reindex(all_combinations, fill_value=0).sort_index()

abc_xyz
AX    581
AY      3
AZ      0
BX    463
BY      8
BZ      0
CX    553
CY     12
CZ      0
Name: count, dtype: int64

## 6. Safety Stock + Reorder Point
Standard formulas:
- **Safety Stock** =  Z × σ_daily × √(lead_time_days)
- **Reorder Point** = (μ_daily × lead_time_days) + Safety Stock

In [35]:
# Service levels now driven by config — no hardcoded values here
yearly_kpi["service_level_z"] = yearly_kpi["abc"].map(ABC_SERVICE_LEVELS)

yearly_kpi["safety_stock"]   = (yearly_kpi["service_level_z"] * yearly_kpi["std_daily_demand"] * np.sqrt(yearly_kpi["lead_time_days"])).round().astype(int)
yearly_kpi["reorder_point"]  = ((yearly_kpi["avg_daily_demand"] * yearly_kpi["lead_time_days"]) + yearly_kpi["safety_stock"]).round().astype(int)
yearly_kpi["days_of_supply"] = (yearly_kpi["current_stock"] / yearly_kpi["avg_daily_demand"]).replace([np.inf, -np.inf], np.nan).round(1)

## 7. Health status flag
- 🔴 **STOCKOUT**     — current_stock = 0
- 🟠 **REORDER NOW**  — current_stock ≤ ROP
- 🟡 **LOW**          — current_stock ≤ 1.5 × ROP
- 🟢 **HEALTHY**      — current_stock > 1.5 × ROP
- ⚪ **OVERSTOCK**    — current_stock > 4 × ROP  (capital tied up)

In [36]:
def status(row):
    cur, rop = row["current_stock"], row["reorder_point"]
    if cur == 0:                                              return "STOCKOUT"
    if rop == 0:                                              return "HEALTHY"
    if cur <= rop:                                            return "REORDER NOW"
    if cur <= STATUS_THRESHOLDS["low_multiplier"] * rop:      return "LOW"
    if cur > STATUS_THRESHOLDS["overstock_multiplier"] * rop: return "OVERSTOCK"
    return "HEALTHY"

yearly_kpi["status"] = yearly_kpi.apply(status, axis=1)
yearly_kpi["status"].value_counts()

status
REORDER NOW    1101
HEALTHY         258
LOW             242
OVERSTOCK        19
Name: count, dtype: int64

## 8. Final KPI table — preview & save

In [37]:
# Define output paths
OUT_KPI      = "E:/Inventory Optimization/output/inventory_kpis_sku_country.csv"
OUT_COUNTRY  = "E:/Inventory Optimization/output/country_summary.csv"
OUT_REORDERS = "E:/Inventory Optimization/output/sku_country_needs_reorder.csv"

# ← ADD THIS BLOCK HERE
daily_kpi["year"] = daily_kpi["date"].dt.year

daily_max_dates = (
    daily_kpi.groupby(["sku_id", "country", "year"])["date"]
         .max()
         .reset_index()
         .rename(columns={"date": "latest_date"})
)

yearly_kpi = yearly_kpi.merge(
    daily_max_dates,
    on=["sku_id", "country", "year"],
    how="left"
)

print(yearly_kpi.columns.tolist())



['sku_id', 'country', 'year', 'total_units', 'total_revenue', 'avg_daily_demand', 'std_daily_demand', 'active_days', 'stockout_days', 'current_stock', 'lead_time_days', 'unit_cost', 'cv_demand', 'stockout_rate', 'sku_name', 'category', 'brand', 'abc', 'xyz', 'abc_xyz', 'service_level_z', 'safety_stock', 'reorder_point', 'days_of_supply', 'status', 'latest_date']


In [39]:

# Then your cols list with year and latest_date added
cols = [
    "sku_id", "sku_name", "category", "brand", "country", "year",  
    "total_units", "total_revenue", "avg_daily_demand", "std_daily_demand",
    "cv_demand", "abc", "xyz", "abc_xyz", "lead_time_days", "safety_stock",
    "reorder_point", "current_stock", "days_of_supply", "stockout_days",
    "stockout_rate", "status", "latest_date",                        
]

missing = [c for c in cols if c not in yearly_kpi.columns]
print("Missing columns:", missing)   # should print []



kpi_out = yearly_kpi[cols].copy()

# Sort: most recent year first, then by country, then by revenue importance
kpi_out = kpi_out.sort_values(
    ["year", "country", "abc", "total_revenue"],
    ascending=[False, True, True, False]
)

kpi_out["avg_daily_demand"] = kpi_out["avg_daily_demand"].round(2)
kpi_out["std_daily_demand"] = kpi_out["std_daily_demand"].round(2)
kpi_out["total_revenue"]    = kpi_out["total_revenue"].round(2)

kpi_out.to_csv(OUT_KPI, index=False)
kpi_out.head()



Missing columns: []


,sku_id,sku_name,category,brand,country,year,total_units,total_revenue,avg_daily_demand,std_daily_demand,...,abc_xyz,lead_time_days,safety_stock,reorder_point,current_stock,days_of_supply,stockout_days,stockout_rate,status,latest_date
1,SKU0027,BrandC Chips,Snacks,BrandC,Austria,2023,47329,628585.42,129.60,58.97,...,AX,6.385714,347,1175,314,2.4,9,0.025,REORDER NOW,2023-12-31
5,SKU0018,BrandF Water,Beverages,BrandF,Austria,2023,38472,558613.44,105.36,37.21,...,AX,6.524616,221,908,336,3.2,7,0.019,REORDER NOW,2023-12-31
8,SKU0092,BrandB Soap,Personal Care,BrandB,Austria,2023,37196,518884.20,101.91,36.83,...,AX,6.490988,219,880,325,3.2,15,0.041,REORDER NOW,2023-12-31
12,SKU0057,BrandC Milk,Dairy,BrandC,Austria,2023,33856,474661.12,92.79,30.94,...,AX,6.612538,185,799,375,4.0,7,0.019,REORDER NOW,2023-12-31
14,SKU0099,BrandC Toothpaste,Personal Care,BrandC,Austria,2023,44994,472312.39,123.31,58.51,...,AX,6.394624,345,1134,327,2.7,12,0.033,REORDER NOW,2023-12-31


## 9. Country-level rollup

In [40]:
country_summary = (
    kpi_out.groupby("country", observed=True)
       .agg(
           skus              = ("sku_id", "nunique"),
           total_revenue     = ("total_revenue", "sum"),
           total_units       = ("total_units", "sum"),
           avg_stockout_rate = ("stockout_rate", "mean"),
           reorder_now       = ("status", lambda s: (s == "REORDER NOW").sum()),
           stockout_skus     = ("status", lambda s: (s == "STOCKOUT").sum()),
           overstock_skus    = ("status", lambda s: (s == "OVERSTOCK").sum()),
       )
       .round(3)
       .sort_values("total_revenue", ascending=False)
)
country_summary.to_csv(OUT_COUNTRY)

country_summary

,skus,total_revenue,total_units,avg_stockout_rate,reorder_now,stockout_skus,overstock_skus
country,,,,,,,
Germany,98,52966750.75,7165795,0.030,208,0,0
Austria,80,43020025.97,5742174,0.031,171,0,0
France,80,42601612.57,6014974,0.031,164,0,1
Poland,80,42385262.52,5860180,0.030,176,0,4
Spain,100,40825952.31,5547641,0.029,191,0,7
Italy,102,39547083.11,5440127,0.030,191,0,7


## 10. Action list — SKUs that need reordering now

In [41]:
needs_reorder = (
    kpi_out[kpi_out["status"].isin(["STOCKOUT", "REORDER NOW"])]
    .sort_values(["abc", "total_revenue"], ascending=[True, False])
)
needs_reorder.to_csv(OUT_REORDERS, index=False)
print(f"{len(needs_reorder):,} SKU × Country pairs need reordering → {OUT_REORDERS}")


needs_reorder.head(30)



1,101 SKU × Country pairs need reordering → E:/Inventory Optimization/output/sku_country_needs_reorder.csv


,sku_id,sku_name,category,brand,country,year,total_units,total_revenue,avg_daily_demand,std_daily_demand,...,abc_xyz,lead_time_days,safety_stock,reorder_point,current_stock,days_of_supply,stockout_days,stockout_rate,status,latest_date
0,SKU0027,BrandC Chips,Snacks,BrandC,Austria,2022,47884,634485.26,131.18,60.01,...,AX,6.459741,355,1202,353,2.7,12,0.033,REORDER NOW,2022-12-31
480,SKU0027,BrandC Chips,Snacks,BrandC,Germany,2022,47604,632212.80,130.27,54.90,...,AX,6.581445,328,1185,402,3.1,5,0.014,REORDER NOW,2022-12-31
1080,SKU0027,BrandC Chips,Snacks,BrandC,Poland,2021,47054,630669.75,129.03,57.77,...,AX,6.534729,344,1187,371,2.9,5,0.014,REORDER NOW,2021-12-31
1,SKU0027,BrandC Chips,Snacks,BrandC,Austria,2023,47329,628585.42,129.60,58.97,...,AX,6.385714,347,1175,314,2.4,9,0.025,REORDER NOW,2023-12-31
1081,SKU0027,BrandC Chips,Snacks,BrandC,Poland,2022,46850,620660.73,128.46,58.47,...,AX,6.509121,348,1184,465,3.6,9,0.025,REORDER NOW,2022-12-31
2,SKU0027,BrandC Chips,Snacks,BrandC,Austria,2021,46003,618833.98,126.04,54.62,...,AX,6.651203,328,1166,322,2.6,11,0.030,REORDER NOW,2021-12-31
1082,SKU0027,BrandC Chips,Snacks,BrandC,Poland,2023,45947,610574.60,125.93,56.72,...,AX,6.275269,331,1121,346,2.7,8,0.022,REORDER NOW,2023-12-31
481,SKU0027,BrandC Chips,Snacks,BrandC,Germany,2021,44833,601440.83,122.88,54.49,...,AX,6.542665,325,1129,233,1.9,11,0.030,REORDER NOW,2021-12-31
482,SKU0027,BrandC Chips,Snacks,BrandC,Germany,2023,44884,596686.38,122.99,57.80,...,AX,6.461630,342,1137,213,1.7,11,0.030,REORDER NOW,2023-12-31
1083,SKU0018,BrandF Water,Beverages,BrandF,Poland,2022,40229,584125.08,110.17,35.14,...,AX,6.477215,208,922,342,3.1,8,0.022,REORDER NOW,2022-12-31


## 11. ABC × XYZ matrix (count of SKU × Country pairs)

In [42]:
matrix = (
    kpi_out.pivot_table(index="abc", columns="xyz", values="sku_id", aggfunc="count", fill_value=0)
       .reindex(index=["A","B","C"], columns=["X","Y","Z"], fill_value=0)
)
matrix.loc["Total"] = matrix.sum()
matrix["Total"]     = matrix.sum(axis=1)
matrix

xyz,X,Y,Z,Total
abc,,,,
A,581,3,0,584
B,463,8,0,471
C,553,12,0,565
Total,1597,23,0,1620
